## Property health baseline: `samples.wanderbricks.reviews`
1. Table profile and data quality
2. Monthly review volume and average rating
3. Rating distribution
4. Properties whose average rating is dropping month over month

Notes:
* Data covers 2023-03-11 to 2025-07-31, so there are no reviews after July 2025.
* Deleted reviews (`is_deleted = true`, which have no rating) are left out of every rating metric.
* `review_id` only takes values 1–1000 and repeats about 100 times each, so it can't be used as a key. Every row is a separate review, and all rows are distinct.
* **Dropping month over month** means: the property has at least 4 months with reviews, and its monthly average rating fell in each of its last 3 month-to-month comparisons. Comparisons use months that have reviews, because most properties get under 2 reviews a month.

In [0]:
%sql
SELECT
  count(*)                                   AS total_rows,
  count(DISTINCT review_id)                  AS distinct_review_ids,
  count(DISTINCT property_id)                AS distinct_properties,
  count(DISTINCT user_id)                    AS distinct_users,
  count(DISTINCT booking_id)                 AS distinct_bookings,
  count_if(is_deleted)                       AS deleted_reviews,
  count_if(rating IS NULL)                   AS null_ratings,
  count_if(comment IS NULL OR trim(comment) = '') AS empty_comments,
  count_if(updated_at <> created_at)         AS edited_reviews,
  min(rating)                                AS min_rating,
  max(rating)                                AS max_rating,
  round(avg(rating), 3)                      AS avg_rating,
  round(stddev(rating), 3)                   AS stddev_rating,
  percentile_approx(rating, array(0.25, 0.5, 0.75)) AS rating_quartiles,
  min(created_at)                            AS first_review,
  max(created_at)                            AS last_review
FROM samples.wanderbricks.reviews

In [0]:
%sql
SELECT
  date_trunc('month', created_at) AS month,
  count(*)                        AS reviews,
  count(DISTINCT property_id)     AS properties_reviewed,
  round(avg(rating), 3)           AS avg_rating
FROM samples.wanderbricks.reviews
WHERE NOT is_deleted
GROUP BY ALL
ORDER BY month

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Ratings are 1.0-5.0 in 0.1 steps; half-star buckets [1.0,1.5) ... [4.5,5.0]
SELECT
  least(floor(rating * 2) / 2, 4.5)                AS rating_bucket,
  count(*)                                         AS reviews,
  round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_of_reviews
FROM samples.wanderbricks.reviews
WHERE NOT is_deleted
GROUP BY ALL
ORDER BY rating_bucket

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH monthly AS (
  SELECT property_id,
         date_trunc('month', created_at) AS month,
         avg(rating) AS avg_rating,
         count(*)    AS reviews
  FROM samples.wanderbricks.reviews
  WHERE NOT is_deleted
  GROUP BY ALL
),
lagged AS (
  SELECT *,
         lag(avg_rating) OVER (PARTITION BY property_id ORDER BY month)    AS prev_avg,
         row_number()    OVER (PARTITION BY property_id ORDER BY month DESC) AS rn_desc,
         count(*)        OVER (PARTITION BY property_id)                    AS active_months
  FROM monthly
),
flagged AS (
  SELECT property_id
  FROM lagged
  WHERE active_months >= 4
  GROUP BY property_id
  HAVING count_if(rn_desc <= 3 AND avg_rating < prev_avg) = 3   -- last 3 MoM changes all negative
)
SELECT
  l.property_id,
  max(l.active_months)                                   AS active_months,
  sum(l.reviews)                                         AS total_reviews,
  round(max(l.avg_rating) FILTER (WHERE l.rn_desc = 4), 2) AS rating_before_decline,
  round(max(l.avg_rating) FILTER (WHERE l.rn_desc = 1), 2) AS latest_month_rating,
  round(max(l.avg_rating) FILTER (WHERE l.rn_desc = 1)
      - max(l.avg_rating) FILTER (WHERE l.rn_desc = 4), 2) AS total_drop,
  max(l.month)                                           AS latest_review_month,
  transform(array_sort(collect_list(struct(l.month AS m, round(l.avg_rating, 2) AS r))), x -> x.r) AS monthly_avg_history
FROM lagged l
JOIN flagged f USING (property_id)
GROUP BY l.property_id
ORDER BY total_drop, latest_review_month DESC